In [1]:
%cd Negation

/home/amirhossein.hajimohammadrezaie/ECOR_extended/Negation


In [1]:
import clip
import torch
import numpy as np
import pandas as pd
import ast
from transformers import CLIPTokenizer
from torch.utils.data import Dataset, DataLoader, default_collate, Subset, random_split
from tqdm import tqdm
import datasets
from peft import LoraConfig, get_peft_model
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn as nn
import seaborn as sns
import os

In [44]:
repharased_df = pd.read_csv("data/images/VOC2007_mcq_llama3.1_rephrased.csv")
templated_df = pd.read_csv("data/images/VOC2007_mcq.csv")

repharased_paths = repharased_df['image_path'].apply(lambda s: os.path.basename(s))
templated_paths = templated_df['image_path'].apply(lambda s: os.path.basename(s))

In [2]:
repharased_df = pd.read_csv("data/images/COCO_val_negated_retrieval_llama3.1_rephrased_affneg_true.csv")
repharased_df['captions'] = repharased_df['captions'].apply(lambda caps: eval(caps))

templated_df = pd.read_csv("data/images/COCO_val_negated_retrieval_template.csv")
templated_df['captions'] = templated_df['captions'].apply(lambda caps: eval(caps))



repharased_paths = repharased_df['filepath'].apply(lambda s: os.path.basename(s))
templated_paths = templated_df['filepath'].apply(lambda s: os.path.basename(s))

In [2]:
def process_MCQ_caption(s: str):
    aff = ""
    neg = ""

    if " not " in s:
        if " does " in s:
            neg = s.split(" does not include ")[1]

        else:
            neg = s.split(" not ")[1]

    
    if " does not " not in s:
        if " but " in s:
            aff_cand = s.split(" but ")[0]
            aff = aff_cand.split(" includes ")[1]
        else:
            try:
                aff = s.split(" includes ")[1]
            except:
                Exception("Error last ....")


    return aff, neg


def preprocess_retreival_caption(cap: str):

    pos_cap = ""
    neg_cap = ""

    idx = cap.find('There is no ')
    assert idx != -1
    
    if idx == 0:
        neg_cap = cap.split('. ')[0]
        neg_cap = neg_cap.split('There is no ')[1]
        neg_cap = neg_cap.split(' in the image')[0]

        pos_cap = '. '.join(cap.split('. ')[1:])
    
    else:
        neg_cap = cap[idx:]
        neg_cap = neg_cap.split('There is no ')[1]
        neg_cap = neg_cap.split(' in the image')[0]

        pos_cap = cap[:idx]

    return pos_cap.strip(), neg_cap.strip()


In [3]:
mcq_repharased_df = pd.read_csv('data/images/VOC2007_mcq_llama3.1_rephrased.csv')
mcq_repharased_df['repharased_caption'] = mcq_repharased_df.apply(lambda row: [row['caption_0'], row['caption_1'], row['caption_2'], row['caption_3']], axis=1)
mcq_repharased_caps = mcq_repharased_df['repharased_caption'].explode(ignore_index=True)

mcq_templated_df = pd.read_csv('data/images/VOC2007_mcq.csv')
mcq_templated_df['caption_0'] = mcq_templated_df['caption_0'].apply(lambda s: f"### Positive Part\n{process_MCQ_caption(s)[0]}\n\n### Negative Part\n{process_MCQ_caption(s)[1]}")
mcq_templated_df['caption_1'] = mcq_templated_df['caption_1'].apply(lambda s: f"### Positive Part\n{process_MCQ_caption(s)[0]}\n\n### Negative Part\n{process_MCQ_caption(s)[1]}")
mcq_templated_df['caption_2'] = mcq_templated_df['caption_2'].apply(lambda s: f"### Positive Part\n{process_MCQ_caption(s)[0]}\n\n### Negative Part\n{process_MCQ_caption(s)[1]}")
mcq_templated_df['caption_3'] = mcq_templated_df['caption_3'].apply(lambda s: f"### Positive Part\n{process_MCQ_caption(s)[0]}\n\n### Negative Part\n{process_MCQ_caption(s)[1]}")
mcq_templated_df['templated_caption'] = mcq_templated_df.apply(lambda row: [row['caption_0'], row['caption_1'], row['caption_2'], row['caption_3']], axis=1)
mcq_templated_caps = mcq_templated_df['templated_caption'].explode(ignore_index=True)
mcq_dataset = pd.concat([mcq_repharased_caps, mcq_templated_caps], axis=1)

retreival_repharased_df = pd.read_csv('data/images/COCO_val_negated_retrieval_llama3.1_rephrased_affneg_true.csv')
retreival_repharased_df['repharased_caption'] = retreival_repharased_df['captions'].apply(lambda caps: eval(caps))
retreival_repharased_caps = retreival_repharased_df['repharased_caption'].explode(ignore_index=True)


retreival_templated_df = pd.read_csv('data/images/COCO_val_negated_retrieval_template.csv')
retreival_templated_df['templated_caption'] = retreival_templated_df['captions'].apply(lambda caps: eval(caps))
retreival_templated_caps = retreival_templated_df['templated_caption'].explode(ignore_index=True)
retreival_templated_caps = retreival_templated_caps.apply(lambda cap: f"### Positive Part\n{preprocess_retreival_caption(cap)[0]}\n\n### Negative Part\n{preprocess_retreival_caption(cap)[1]}")
retreival_dataset = pd.concat([retreival_repharased_caps, retreival_templated_caps], axis=1)


dataset = pd.concat([mcq_dataset, retreival_dataset], axis=0).reset_index(drop=True)


In [9]:
len(dataset) * 0.8

36110.4

In [7]:
idx = np.random.randint(len(dataset))
print(dataset.iloc[idx]['repharased_caption'])
print("-------------------------------------------")
print(dataset.iloc[idx]['templated_caption'])


A stuffed bunny is gripping a toothbrush, but there's no cup to be found.
-------------------------------------------
### Positive Part
A stuffed bunny is holding a tooth brush.

### Negative Part
cup


In [102]:
rephrased_df = pd.read_csv('data/images/VOC2007_mcq_llama3.1_rephrased.csv')
rephrased_df['repharased_caption'] = rephrased_df.apply(lambda row: [row['caption_0'], row['caption_1'], row['caption_2'], row['caption_3']], axis=1)
repharased_caps = rephrased_df['repharased_caption'].explode(ignore_index=True)

template_df = pd.read_csv('data/images/VOC2007_mcq.csv')
template_df['caption_0'] = template_df['caption_0'].apply(lambda s: f"### Positive Part\n{process_caption(s)[0]}\n\n### Negative Part\n{process_caption(s)[1]}")
template_df['caption_1'] = template_df['caption_1'].apply(lambda s: f"### Positive Part\n{process_caption(s)[0]}\n\n### Negative Part\n{process_caption(s)[1]}")
template_df['caption_2'] = template_df['caption_2'].apply(lambda s: f"### Positive Part\n{process_caption(s)[0]}\n\n### Negative Part\n{process_caption(s)[1]}")
template_df['caption_3'] = template_df['caption_3'].apply(lambda s: f"### Positive Part\n{process_caption(s)[0]}\n\n### Negative Part\n{process_caption(s)[1]}")
template_df['templated_caption'] = template_df.apply(lambda row: [row['caption_0'], row['caption_1'], row['caption_2'], row['caption_3']], axis=1)
templated_caps = template_df['templated_caption'].explode(ignore_index=True)

dataset = pd.concat([repharased_caps, templated_caps], axis=1)

dataset = datasets.Dataset.from_pandas(dataset)
subsets = dataset.train_test_split(train_size=0.8, seed=42, shuffle=True)
trainset, valset = subsets['train'], subsets['test']

In [105]:
def preprocess_function(example):
    return {
        "prompt": [{"role": "user", "content": example["repharased_caption"]}],
        "completion": [
            {"role": "assistant", "content": f"{example['templated_caption']}"}
        ],
    }


trainset = trainset.map(preprocess_function, remove_columns=trainset.column_names)
valset = valset.map(preprocess_function, remove_columns=valset.column_names)

Map:   0%|          | 0/16099 [00:00<?, ? examples/s]

Map:   0%|          | 0/4025 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", device_map='auto', torch_dtype=torch.float16)

lora_cfg = LoraConfig(
    r=8, 
    lora_alpha=16, 
    lora_dropout=0.05,
    bias="none", 
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)

args = SFTConfig(
    output_dir="/home/amirhossein.hajimohammadrezaie/ECOR_extended/Generative-Monoculture-Mitigation/mistral-7B-sft-Negation",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    max_length=512,
    fp16=False,
    bf16=True,
    optim='paged_adamw_32bit',
    report_to='wandb',
    completion_only_loss=True
)

trainer = SFTTrainer(model, args, train_dataset=trainset, eval_dataset=valset, processing_class=tokenizer, peft_config=lora_cfg)
trainer.train()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

'outputs = model.generate(**inputs, max_new_tokens=40)\nprint(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))'